# 11 · Producción: código limpio, API, Docker, MLOps y cloud (≈1 h)

La oferta: *"código limpio, modular, mantenible y eficiente"*, *"integración de modelos con otros componentes de
software"*, *"Docker, APIs y MLOps"*, *"Azure, AWS"*, *"desplegando modelos en producción"*, *"documentación de
experimentos"*.

En esta carpeta hay un mini-servicio real:
```
11_produccion/
  app/pipeline.py      ← lógica: config tipada, lectura de vídeo, detector+tracker, salida estructurada
  app/main.py          ← API FastAPI: /health, /track (sube un vídeo → JSON de tracks)
  tests/test_api.py    ← tests con pytest + TestClient
  Dockerfile           ← imagen del servicio
  requirements-app.txt
```
Ábrelos y léelos (son cortos). Aquí los ejercitamos y repasamos la teoría.

In [ ]:
# ⚙️ SETUP · ejecuta esta celda primero. Hace el notebook independiente: funciona en local (dentro del repo)
# y en Google Colab (instala lo que falte, clona el repo y descarga los datos a ./data).
import sys, subprocess, importlib.util, urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
PIP = {"ultralytics": "ultralytics", "fastapi": "fastapi", "httpx": "httpx", "python-multipart": "multipart", "pytest": "pytest"}   # paquete pip → módulo que importa
missing = [p for p, m in PIP.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Raíz del repo: la carpeta que contiene vision_utils/ (subiendo desde aquí, o el clon de Colab)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "vision-bootcamp"]
             if (p / "vision_utils").is_dir()), None)
if ROOT is None:   # este notebook usa código del repo (vision_utils/ o 11_produccion/) → clonarlo
    ROOT = Path.cwd() / "vision-bootcamp"
    url = "https://github.com/jcandel01/vision-bootcamp.git"
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", url, str(ROOT)], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("No se pudo clonar el repo: " + r.stderr)
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
DATA.mkdir(exist_ok=True)
if not (DATA / "people.mp4").exists():   # vídeo de ejemplo (cámara fija, ~14 s, 1920×1080)
    urllib.request.urlretrieve("https://media.roboflow.com/supervision/video-examples/people-walking.mp4",
                               DATA / "people.mp4")
# Los pesos YOLO (yolo11n.pt…) no hace falta descargarlos: Ultralytics los baja solo la primera vez que se usan.
print("ROOT:", ROOT, "| DATA:", DATA, "| Colab:", IN_COLAB)

In [ ]:
import sys, json, subprocess
from pathlib import Path
HERE = ROOT / "11_produccion"
sys.path.insert(0, str(HERE))

## 1. Código limpio en un proyecto de visión

Lo que un revisor mira (y lo que deberías decir que haces):
- **Separar responsabilidades**: E/S de vídeo, modelo, tracking, geometría, exportación de datos, API → módulos
  distintos con interfaces pequeñas. El notebook es para explorar; el código de producción vive en paquetes.
- **Configuración explícita y tipada** (`dataclass`/Pydantic, YAML/Hydra) en vez de constantes sueltas; la
  config de cada experimento se guarda junto a sus resultados.
- **Tipos** (`type hints`), docstrings breves, nombres claros, funciones puras donde se pueda (más fáciles de testear).
- **Tests**: unitarios de la lógica (IoU, NMS, homografía, métricas — todo lo que implementaste en este curso),
  tests de integración del pipeline con un vídeo corto, y **tests de regresión de métricas** (que el HOTA en el
  set de validación no baje de X tras un cambio).
- **Logging** (no `print`), errores con mensajes útiles, recursos liberados (`try/finally`, *context managers*).
- Herramientas: `ruff` (lint+formato), `mypy`, `pre-commit`, CI (GitHub Actions) que ejecuta tests.
- **Rendimiento**: vectorizar con NumPy, evitar bucles Python por píxel, cargar el modelo una vez, no copiar
  frames innecesariamente.

In [ ]:
print((HERE / "app" / "pipeline.py").read_text()[:2500])

## 2. La API

Decisiones de diseño que conviene saber justificar:
- **Cargar el modelo al arrancar** (`lifespan`), no en cada petición.
- **Endpoints síncronos para trabajo pesado**: FastAPI los ejecuta en un *threadpool*; el modelo con estado
  (tracker) se protege con un lock. Para vídeos largos, mejor **asíncrono con cola**: `POST /jobs` devuelve un
  `job_id`, un worker procesa, `GET /jobs/{id}` devuelve el estado/resultado (o se escribe en S3/Blob y se
  notifica).
- **Validación de entrada** (tipo de fichero, límites) y códigos HTTP correctos (415, 400…).
- **Esquema de salida estable y versionado** (el de `PipelineResult.to_dict()`).
- `/health` para que el orquestador (Kubernetes, ECS) sepa si el servicio está vivo y listo.

Probamos la API sin levantar servidor, con el `TestClient`:

In [ ]:
from fastapi.testclient import TestClient
from app.main import app

with TestClient(app) as client:
    print(client.get("/health").json())
    with open(DATA / "people.mp4", "rb") as f:
        r = client.post("/track?max_frames=50", files={"file": ("people.mp4", f, "video/mp4")})
body = r.json()
print({k: v for k, v in body.items() if k != "points"})
print(json.dumps(body["points"][:2], indent=2))

Para levantarla de verdad (en una terminal):
```bash
cd 11_produccion && uvicorn app.main:app --port 8000
# http://localhost:8000/docs  → Swagger interactivo
curl -F "file=@../data/people.mp4" "http://localhost:8000/track?max_frames=100" | head -c 500
```

## 3. Tests

In [ ]:
print(subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"], cwd=HERE,
                     capture_output=True, text=True).stdout[-600:])

## 4. Docker

Lee el `Dockerfile`. Puntos clave:
- Imagen base ligera (`python:3.12-slim`) o, para GPU, una de NVIDIA (`nvcr.io/nvidia/pytorch`,
  `nvidia/cuda:*-runtime`) + `docker run --gpus all` (requiere *NVIDIA Container Toolkit* en el host).
- **Orden de capas**: dependencias antes que el código → las reconstrucciones solo rehacen la última capa.
- Dependencias de sistema de OpenCV/FFmpeg (`libgl1`, `libglib2.0-0`, `ffmpeg`), `opencv-python-headless`.
- Pesos fuera de la imagen (volumen, o descarga desde un registro de modelos al arrancar) si cambian a menudo.
- Usuario sin privilegios, sin secretos en la imagen (variables de entorno / gestor de secretos).
- **Multi-stage builds** para imágenes más pequeñas; fijar versiones para builds reproducibles.
```bash
docker build -t vision-api 11_produccion
docker run -p 8000:8000 -v $PWD/data:/models vision-api
```

## 5. MLOps: el ciclo de vida del modelo

| Necesidad | Herramientas |
|---|---|
| Versionado de datos | DVC, Roboflow versions, LakeFS, o S3 + manifest con hashes |
| Seguimiento de experimentos (config, métricas, artefactos) | **MLflow**, Weights & Biases, ClearML (Ultralytics se integra con todos) |
| Registro de modelos (versiones, stage: staging/prod) | MLflow Model Registry, W&B, SageMaker / Azure ML registry |
| Entrenamiento reproducible | config + semilla + versión de datos + imagen Docker; pipelines (Kubeflow, Airflow, SageMaker Pipelines, Azure ML) |
| Servir modelos | FastAPI propio, **Triton Inference Server** (dynamic batching, TensorRT/ONNX), TorchServe, KServe |
| CI/CD | GitHub Actions / GitLab CI: tests, lint, build de imagen, despliegue; *gates* de métricas |
| Monitorización | latencia, FPS, errores; **drift** de datos (brillo, nº de detecciones, distribución de scores), muestreo de predicciones para revisión humana |
| Mejora continua | casos difíciles de producción → anotar → reentrenar → evaluar contra el modelo actual → desplegar si mejora |

**Documentar experimentos** (lo pide la oferta): cada experimento con su hipótesis, config, versión de datos,
métricas (mAP, HOTA, IDF1, FPS), conclusiones y decisión. Un *experiment log* en Markdown/Notion + MLflow basta.

## 6. Cloud (AWS / Azure) en dos frases

| Pieza | AWS | Azure |
|---|---|---|
| Almacenamiento de vídeos/datasets | S3 | Blob Storage |
| GPUs para entrenar/inferir | EC2 (g5/g6 = A10G/L4, p4/p5 = A100/H100) | NC/ND-series VMs |
| Contenedores | ECR + ECS/EKS (Kubernetes) | ACR + AKS |
| Plataforma ML gestionada | SageMaker | Azure Machine Learning |
| Colas / eventos | SQS, SNS, Kinesis | Service Bus, Event Hubs |
| Serverless | Lambda (sin GPU) | Functions |

Patrón típico de procesamiento de vídeo por lotes: el vídeo llega a **S3/Blob** → evento → mensaje en **cola** →
**workers con GPU** en Kubernetes (autoescalado según la cola) procesan → resultados (Parquet/JSON) a S3 y a la
base de datos de la plataforma → notificación. En tiempo real: ingestión RTSP en el *edge* (Jetson + DeepStream)
o en GPUs cloud, metadatos por Kafka/Kinesis.

## 7. C++
- En visión, C++ aparece en: inferencia con TensorRT/ONNX Runtime C++ API, plugins de DeepStream/GStreamer,
  OpenCV C++, y partes críticas de rendimiento. Python ↔ C++ con **pybind11**.
- Conceptos que conviene tener frescos: RAII y *smart pointers* (`std::unique_ptr`, `std::shared_ptr`),
  referencias vs punteros, `const`, `std::vector`, move semantics, compilar con CMake.
```cpp
#include <opencv2/opencv.hpp>
int main() {
    cv::VideoCapture cap("people.mp4");
    cv::Mat frame;
    while (cap.read(frame)) {
        cv::Mat blob = cv::dnn::blobFromImage(frame, 1.0 / 255, cv::Size(640, 640), cv::Scalar(), true, false);
        // net.setInput(blob); auto out = net.forward();   // cv::dnn::readNetFromONNX("yolo11n.onnx")
    }
}
```